# Nettoyage & enrichissement des boxscores (minutes, DNP, totaux, ratios)

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
from src.config import *
from src.utils import *
import datetime

In [2]:
#store start time of notebook
start_time = datetime.datetime.now()
print("Start time: ", start_time)

Start time:  2025-05-23 11:55:09.411192


## 1 Nettoyer la colonne MIN dans les boxscores

In [3]:



# Si tu utilises plusieurs fichiers, adapte avec concat, ici sur un seul batch pour la démo :
boxscores_file = get_latest_file(DATA_BOXSCORES_BATCHES_MERGED_DIR)  # Ou DATA_PLAYERS_DIR selon ton code
games_file = get_latest_file(DATA_GAMES_DIR)  # Ou DATA_PLAYERS_DIR selon ton code

df_boxscores = pd.read_csv(boxscores_file, low_memory=False)
df_games = pd.read_csv(games_file, low_memory=False)

print(df_games.columns)

# 3. Merge pour ajouter GAME_DATE à chaque ligne de boxscore
if 'GAME_DATE' not in df_boxscores.columns:
    # Pour éviter les doublons ou les merges en cascade si tu relances le code
    if 'GAME_DATE' in df_games.columns:
        df_boxscores = df_boxscores.merge(
            df_games[['GAME_ID', 'GAME_DATE']],
            on='GAME_ID',
            how='left'
        )
        # Conversion en datetime
        df_boxscores['GAME_DATE'] = pd.to_datetime(df_boxscores['GAME_DATE'])
    else:
        print("No GAME_DATE column found in games_df. Please check your merge operation.")

# Fonction de conversion “mm:ss” -> float minutes
def convert_minutes(val):
    if pd.isna(val) or val in ['DNP', '']:
        return 0.0
    if isinstance(val, float) or isinstance(val, int):
        return float(val)
    try:
        parts = str(val).split(':')
        if len(parts) == 2:
            minutes = int(parts[0])
            seconds = int(parts[1])
            return minutes + seconds/60
        else:
            # Cas où c'est déjà un float/int
            return float(val)
    except:
        return 0.0

df_boxscores['MINUTES_PLAYED'] = df_boxscores['MIN'].apply(convert_minutes)

print(df_boxscores[['PLAYER_NAME', 'MIN', 'MINUTES_PLAYED']].head(10))
print(df_boxscores['MINUTES_PLAYED'].describe())


Index(['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID',
       'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT',
       'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB',
       'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS', 'SEASON'],
      dtype='object')
         PLAYER_NAME    MIN  MINUTES_PLAYED
0  Clifford Robinson  42:12       42.200000
1  Clifford Robinson  42:12       42.200000
2       Shawn Marion  28:07       28.116667
3       Shawn Marion  28:07       28.116667
4       Chris Dudley  14:37       14.616667
5       Chris Dudley  14:37       14.616667
6         Mario Elie  10:23       10.383333
7         Mario Elie  10:23       10.383333
8         Jason Kidd  44:46       44.766667
9         Jason Kidd  44:46       44.766667
count    1.707592e+06
mean     1.822732e+01
std      1.394997e+01
min      0.000000e+00
25%      3.000000e+00
50%      1.903333e+01
75%      3.003333e+01
max      6.496667e+01
Name: MINUTES_PLAYED

In [4]:
df_boxscores

,GAME_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_CITY,PLAYER_ID,PLAYER_NAME,NICKNAME,START_POSITION,COMMENT,MIN,...,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,GAME_DATE,MINUTES_PLAYED
0,20000011,1610612756,PHX,Phoenix,361,Clifford Robinson,Clifford,G,NaN,42:12,...,5.0,3.0,2.0,1.0,1.0,2.0,26.0,-3.0,2000-10-31,42.200000
1,20000011,1610612756,PHX,Phoenix,361,Clifford Robinson,Clifford,G,NaN,42:12,...,5.0,3.0,2.0,1.0,1.0,2.0,26.0,-3.0,2000-10-31,42.200000
2,20000011,1610612756,PHX,Phoenix,1890,Shawn Marion,Shawn,G,NaN,28:07,...,6.0,1.0,1.0,1.0,3.0,3.0,16.0,-12.0,2000-10-31,28.116667
3,20000011,1610612756,PHX,Phoenix,1890,Shawn Marion,Shawn,G,NaN,28:07,...,6.0,1.0,1.0,1.0,3.0,3.0,16.0,-12.0,2000-10-31,28.116667
4,20000011,1610612756,PHX,Phoenix,201,Chris Dudley,Chris,F,NaN,14:37,...,2.0,0.0,0.0,0.0,2.0,3.0,2.0,3.0,2000-10-31,14.616667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1707587,42000406,1610612749,MIL,Milwaukee,1630241,Sam Merrill,Sam,NaN,DNP - Coach's Decision,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-07-20,0.000000
1707588,42000406,1610612749,MIL,Milwaukee,1629670,Jordan Nwora,Jordan,NaN,DNP - Coach's Decision,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-07-20,0.000000
1707589,42000406,1610612749,MIL,Milwaukee,1629670,Jordan Nwora,Jordan,NaN,DNP - Coach's Decision,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-07-20,0.000000
1707590,42000406,1610612749,MIL,Milwaukee,1626253,Axel Toupane,Axel,NaN,DNP - Coach's Decision,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-07-20,0.000000


## Type correction

In [5]:
cols_to_float = [
    'PTS', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'FGM', 'FGA', 'FG3M', 'FG3A', 
    'FTM', 'FTA', 'OREB', 'DREB', 'PLUS_MINUS'
]
for col in cols_to_float:
    if col in df_boxscores.columns:
        df_boxscores[col] = pd.to_numeric(df_boxscores[col], errors='coerce').fillna(0)


 ### Optionnel : repérage/ajout d’un flag “Starter”

In [6]:
df_boxscores['IS_STARTER'] = df_boxscores['START_POSITION'].notna() & (df_boxscores['START_POSITION'] != '')

## Étape 2 — Agrégation de stats par équipe et par match

In [7]:
agg_cols = [
    'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT',
    'FTM', 'FTA', 'FT_PCT',
    'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS', 'PLUS_MINUS', 'MINUTES_PLAYED'
    ]


team_match_stats = df_boxscores.groupby(['GAME_ID', 'TEAM_ID', 'GAME_DATE'])[agg_cols].sum().reset_index()

# Optionnel : rajoute l’équipe adverse dans chaque ligne pour faciliter le merge futur
teams_in_game = df_boxscores.groupby('GAME_ID')['TEAM_ID'].unique().to_dict()
team_match_stats['OPP_TEAM_ID'] = team_match_stats.apply(
    lambda row: [tid for tid in teams_in_game[row['GAME_ID']] if tid != row['TEAM_ID']][0], axis=1
)



In [8]:
team_match_stats


,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,REB,AST,STL,BLK,TO,PF,PTS,PLUS_MINUS,MINUTES_PLAYED,OPP_TEAM_ID
0,20000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,74.0,28.0,12.0,8.0,44.0,60.0,144.0,-290.0,480.000000,1610612755
1,20000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,74.0,54.0,20.0,10.0,26.0,48.0,202.0,290.0,480.000000,1610612752
2,20000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,104.0,32.0,10.0,16.0,38.0,54.0,172.0,40.0,480.000000,1610612751
3,20000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,94.0,48.0,18.0,16.0,24.0,62.0,164.0,-40.0,480.000000,1610612739
4,20000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,74.0,40.0,20.0,18.0,30.0,48.0,194.0,110.0,480.000000,1610612764
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64131,52300131,1610612758,2024-04-16,86.0,196.0,8.150,36.0,78.0,5.796,28.0,...,98.0,56.0,20.0,10.0,16.0,34.0,236.0,240.0,480.033333,1610612744
64132,52300201,1610612741,2024-04-19,70.0,184.0,8.816,26.0,86.0,4.130,16.0,...,76.0,54.0,24.0,12.0,24.0,38.0,182.0,-210.0,480.000000,1610612748
64133,52300201,1610612748,2024-04-19,76.0,164.0,9.182,28.0,66.0,7.888,44.0,...,94.0,52.0,12.0,12.0,30.0,22.0,224.0,210.0,480.000000,1610612741
64134,52300211,1610612740,2024-04-19,88.0,170.0,8.708,14.0,38.0,6.072,20.0,...,90.0,58.0,18.0,16.0,30.0,38.0,210.0,70.0,480.066667,1610612758


##  Renommer les colonnes pour merge

In [9]:
team_cols = [col for col in team_match_stats.columns if col not in ['GAME_ID', 'TEAM_ID', 'OPP_TEAM_ID']]
opp_cols = [f"OPP_{col}" for col in team_cols]

df_team = team_match_stats.copy()
df_opp = team_match_stats.copy()
df_opp = df_opp.rename(
    columns={col: f"OPP_{col}" for col in team_cols}
).rename(
    columns={'TEAM_ID': 'OPP_TEAM_ID', 'OPP_TEAM_ID': 'TEAM_ID'}
)

df_team
df_opp

,GAME_ID,OPP_TEAM_ID,OPP_GAME_DATE,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,...,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TO,OPP_PF,OPP_PTS,OPP_PLUS_MINUS,OPP_MINUTES_PLAYED,TEAM_ID
0,20000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,74.0,28.0,12.0,8.0,44.0,60.0,144.0,-290.0,480.000000,1610612755
1,20000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,74.0,54.0,20.0,10.0,26.0,48.0,202.0,290.0,480.000000,1610612752
2,20000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,104.0,32.0,10.0,16.0,38.0,54.0,172.0,40.0,480.000000,1610612751
3,20000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,94.0,48.0,18.0,16.0,24.0,62.0,164.0,-40.0,480.000000,1610612739
4,20000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,74.0,40.0,20.0,18.0,30.0,48.0,194.0,110.0,480.000000,1610612764
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64131,52300131,1610612758,2024-04-16,86.0,196.0,8.150,36.0,78.0,5.796,28.0,...,98.0,56.0,20.0,10.0,16.0,34.0,236.0,240.0,480.033333,1610612744
64132,52300201,1610612741,2024-04-19,70.0,184.0,8.816,26.0,86.0,4.130,16.0,...,76.0,54.0,24.0,12.0,24.0,38.0,182.0,-210.0,480.000000,1610612748
64133,52300201,1610612748,2024-04-19,76.0,164.0,9.182,28.0,66.0,7.888,44.0,...,94.0,52.0,12.0,12.0,30.0,22.0,224.0,210.0,480.000000,1610612741
64134,52300211,1610612740,2024-04-19,88.0,170.0,8.708,14.0,38.0,6.072,20.0,...,90.0,58.0,18.0,16.0,30.0,38.0,210.0,70.0,480.066667,1610612758


## Merge

In [10]:
match_dataset = pd.merge(
    df_team,
    df_opp[['GAME_ID', 'TEAM_ID'] + opp_cols],
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)


In [13]:
match_dataset 
match_dataset.shape
match_dataset


,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,OPP_DREB,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TO,OPP_PF,OPP_PTS,OPP_PLUS_MINUS,OPP_MINUTES_PLAYED
0,20000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,58.0,74.0,54.0,20.0,10.0,26.0,48.0,202.0,290.0,480.0
1,20000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,46.0,74.0,28.0,12.0,8.0,44.0,60.0,144.0,-290.0,480.0
2,20000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,70.0,94.0,48.0,18.0,16.0,24.0,62.0,164.0,-40.0,480.0
3,20000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,82.0,104.0,32.0,10.0,16.0,38.0,54.0,172.0,40.0,480.0
4,20000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,70.0,88.0,40.0,12.0,2.0,52.0,56.0,172.0,-110.0,480.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64131,42400226,1610612760,2025-05-15,86.0,184.0,9.758,22.0,80.0,4.744,20.0,...,82.0,104.0,54.0,16.0,12.0,28.0,32.0,238.0,120.0,0.0
64132,42400216,1610612738,2025-05-16,62.0,172.0,9.404,24.0,80.0,6.732,14.0,...,80.0,110.0,50.0,14.0,12.0,28.0,46.0,238.0,380.0,0.0
64133,42400216,1610612752,2025-05-16,84.0,182.0,8.478,32.0,90.0,6.500,38.0,...,54.0,72.0,38.0,8.0,6.0,30.0,44.0,162.0,-380.0,0.0
64134,42400227,1610612743,2025-05-18,66.0,168.0,8.318,20.0,88.0,5.016,34.0,...,58.0,88.0,56.0,32.0,6.0,18.0,38.0,250.0,320.0,0.0


In [11]:
match_dataset.columns

Index(['GAME_ID', 'TEAM_ID', 'GAME_DATE', 'FGM', 'FGA', 'FG_PCT', 'FG3M',
       'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST',
       'STL', 'BLK', 'TO', 'PF', 'PTS', 'PLUS_MINUS', 'MINUTES_PLAYED',
       'OPP_TEAM_ID', 'OPP_GAME_DATE', 'OPP_FGM', 'OPP_FGA', 'OPP_FG_PCT',
       'OPP_FG3M', 'OPP_FG3A', 'OPP_FG3_PCT', 'OPP_FTM', 'OPP_FTA',
       'OPP_FT_PCT', 'OPP_OREB', 'OPP_DREB', 'OPP_REB', 'OPP_AST', 'OPP_STL',
       'OPP_BLK', 'OPP_TO', 'OPP_PF', 'OPP_PTS', 'OPP_PLUS_MINUS',
       'OPP_MINUTES_PLAYED'],
      dtype='object')

In [14]:
match_dataset = match_dataset.sort_values(['GAME_DATE', 'GAME_ID']).reset_index(drop=True)
match_dataset


,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,OPP_DREB,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TO,OPP_PF,OPP_PTS,OPP_PLUS_MINUS,OPP_MINUTES_PLAYED
0,20000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,58.0,74.0,54.0,20.0,10.0,26.0,48.0,202.0,290.0,480.0
1,20000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,46.0,74.0,28.0,12.0,8.0,44.0,60.0,144.0,-290.0,480.0
2,20000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,70.0,94.0,48.0,18.0,16.0,24.0,62.0,164.0,-40.0,480.0
3,20000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,82.0,104.0,32.0,10.0,16.0,38.0,54.0,172.0,40.0,480.0
4,20000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,70.0,88.0,40.0,12.0,2.0,52.0,56.0,172.0,-110.0,480.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64131,42400226,1610612760,2025-05-15,86.0,184.0,9.758,22.0,80.0,4.744,20.0,...,82.0,104.0,54.0,16.0,12.0,28.0,32.0,238.0,120.0,0.0
64132,42400216,1610612738,2025-05-16,62.0,172.0,9.404,24.0,80.0,6.732,14.0,...,80.0,110.0,50.0,14.0,12.0,28.0,46.0,238.0,380.0,0.0
64133,42400216,1610612752,2025-05-16,84.0,182.0,8.478,32.0,90.0,6.500,38.0,...,54.0,72.0,38.0,8.0,6.0,30.0,44.0,162.0,-380.0,0.0
64134,42400227,1610612743,2025-05-18,66.0,168.0,8.318,20.0,88.0,5.016,34.0,...,58.0,88.0,56.0,32.0,6.0,18.0,38.0,250.0,320.0,0.0


## Add Win/Loose flag

In [15]:
# Garde l’info du score
match_dataset['IS_WIN'] = (match_dataset['PTS'] > match_dataset['OPP_PTS']).astype(int)
match_dataset['POINT_DIFF'] = match_dataset['PTS'] - match_dataset['OPP_PTS']

match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,OPP_AST,OPP_STL,OPP_BLK,OPP_TO,OPP_PF,OPP_PTS,OPP_PLUS_MINUS,OPP_MINUTES_PLAYED,IS_WIN,POINT_DIFF
0,20000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,54.0,20.0,10.0,26.0,48.0,202.0,290.0,480.0,0,-58.0
1,20000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,28.0,12.0,8.0,44.0,60.0,144.0,-290.0,480.0,1,58.0
2,20000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,48.0,18.0,16.0,24.0,62.0,164.0,-40.0,480.0,1,8.0
3,20000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,32.0,10.0,16.0,38.0,54.0,172.0,40.0,480.0,0,-8.0
4,20000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,40.0,12.0,2.0,52.0,56.0,172.0,-110.0,480.0,1,22.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64131,42400226,1610612760,2025-05-15,86.0,184.0,9.758,22.0,80.0,4.744,20.0,...,54.0,16.0,12.0,28.0,32.0,238.0,120.0,0.0,0,-24.0
64132,42400216,1610612738,2025-05-16,62.0,172.0,9.404,24.0,80.0,6.732,14.0,...,50.0,14.0,12.0,28.0,46.0,238.0,380.0,0.0,0,-76.0
64133,42400216,1610612752,2025-05-16,84.0,182.0,8.478,32.0,90.0,6.500,38.0,...,38.0,8.0,6.0,30.0,44.0,162.0,-380.0,0.0,1,76.0
64134,42400227,1610612743,2025-05-18,66.0,168.0,8.318,20.0,88.0,5.016,34.0,...,56.0,32.0,6.0,18.0,38.0,250.0,320.0,0.0,0,-64.0


## Add IS_HOME flag

In [16]:
mask_home = df_games['MATCHUP'].str.contains('vs\.')
mask_away = df_games['MATCHUP'].str.contains('@')
mask_none = ~(mask_home | mask_away)  # Ni l’un ni l’autre

# Affiche le nombre de cas problématiques (doit être 0 normalement)
print(f"Lignes indéterminées : {mask_none.sum()}")

# Si tu veux voir lesquelles
if mask_none.sum() > 0:
    print(df_games.loc[mask_none, ['GAME_ID', 'TEAM_ABBREVIATION', 'MATCHUP']])


Lignes indéterminées : 0


In [17]:
if 'MATCHUP' in match_dataset.columns:
    match_dataset['IS_HOME'] = match_dataset['MATCHUP'].str.contains('vs\.').astype(int)
else:
    # Faut merge avec df_games pour récupérer MATCHUP
    match_dataset = match_dataset.merge(
        df_games[['GAME_ID', 'TEAM_ID', 'MATCHUP']],
        on=['GAME_ID', 'TEAM_ID'],
        how='left'
    )
    match_dataset['IS_HOME'] = match_dataset['MATCHUP'].str.contains('vs\.').astype(int)
    
#remove MATCHUP column because we don't need it anymore
match_dataset = match_dataset.drop(columns=['MATCHUP'])


In [18]:
match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,OPP_STL,OPP_BLK,OPP_TO,OPP_PF,OPP_PTS,OPP_PLUS_MINUS,OPP_MINUTES_PLAYED,IS_WIN,POINT_DIFF,IS_HOME
0,20000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,20.0,10.0,26.0,48.0,202.0,290.0,480.0,0,-58.0,1
1,20000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,12.0,8.0,44.0,60.0,144.0,-290.0,480.0,1,58.0,0
2,20000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,18.0,16.0,24.0,62.0,164.0,-40.0,480.0,1,8.0,0
3,20000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,10.0,16.0,38.0,54.0,172.0,40.0,480.0,0,-8.0,1
4,20000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,12.0,2.0,52.0,56.0,172.0,-110.0,480.0,1,22.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64131,42400226,1610612760,2025-05-15,86.0,184.0,9.758,22.0,80.0,4.744,20.0,...,16.0,12.0,28.0,32.0,238.0,120.0,0.0,0,-24.0,0
64132,42400216,1610612738,2025-05-16,62.0,172.0,9.404,24.0,80.0,6.732,14.0,...,14.0,12.0,28.0,46.0,238.0,380.0,0.0,0,-76.0,0
64133,42400216,1610612752,2025-05-16,84.0,182.0,8.478,32.0,90.0,6.500,38.0,...,8.0,6.0,30.0,44.0,162.0,-380.0,0.0,1,76.0,1
64134,42400227,1610612743,2025-05-18,66.0,168.0,8.318,20.0,88.0,5.016,34.0,...,32.0,6.0,18.0,38.0,250.0,320.0,0.0,0,-64.0,0


## Rolling winrate on N matchs and global

In [19]:
N_LIST = [5, 10, 25, 50, 100, 200]

for n in N_LIST:
    # Rolling winrate à domicile
    match_dataset[f'ROLL_HOME_WINRATE_{n}'] = (
        match_dataset
        .sort_values(['TEAM_ID', 'GAME_DATE'])
        .groupby('TEAM_ID')
        .apply(lambda df: df['IS_WIN'].where(df['IS_HOME'] == 1).shift(1).rolling(n, min_periods=1).mean())
        .reset_index(level=0, drop=True)
    )

    # Rolling winrate à l'extérieur
    match_dataset[f'ROLL_AWAY_WINRATE_{n}'] = (
        match_dataset
        .sort_values(['TEAM_ID', 'GAME_DATE'])
        .groupby('TEAM_ID')
        .apply(lambda df: df['IS_WIN'].where(df['IS_HOME'] == 0).shift(1).rolling(n, min_periods=1).mean())
        .reset_index(level=0, drop=True)
    )


In [20]:
match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,ROLL_HOME_WINRATE_10,ROLL_AWAY_WINRATE_10,ROLL_HOME_WINRATE_25,ROLL_AWAY_WINRATE_25,ROLL_HOME_WINRATE_50,ROLL_AWAY_WINRATE_50,ROLL_HOME_WINRATE_100,ROLL_AWAY_WINRATE_100,ROLL_HOME_WINRATE_200,ROLL_AWAY_WINRATE_200
0,20000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64131,42400226,1610612760,2025-05-15,86.0,184.0,9.758,22.0,80.0,4.744,20.0,...,0.800000,0.800000,0.785714,0.818182,0.821429,0.772727,0.826923,0.770833,0.806122,0.656863
64132,42400216,1610612738,2025-05-16,62.0,172.0,9.404,24.0,80.0,6.732,14.0,...,0.666667,0.500000,0.636364,0.785714,0.666667,0.807692,0.705882,0.775510,0.764706,0.734694
64133,42400216,1610612752,2025-05-16,84.0,182.0,8.478,32.0,90.0,6.500,38.0,...,0.250000,0.833333,0.615385,0.583333,0.615385,0.625000,0.640000,0.600000,0.660000,0.560000
64134,42400227,1610612743,2025-05-18,66.0,168.0,8.318,20.0,88.0,5.016,34.0,...,0.800000,0.400000,0.538462,0.416667,0.653846,0.458333,0.607843,0.551020,0.712871,0.575758


## Streaks à domicile / extérieur

In [21]:
def calc_streak_home_away(results, is_home):
    streak_home = []
    streak_away = []
    cur_home = cur_away = 0
    for r, h in zip(results, is_home):
        if h == 1:
            if r == 1:
                cur_home += 1
            else:
                cur_home = 0
            streak_home.append(cur_home)
            streak_away.append(cur_away)
        else:
            if r == 1:
                cur_away += 1
            else:
                cur_away = 0
            streak_away.append(cur_away)
            streak_home.append(cur_home)
    return streak_home, streak_away

grouped = match_dataset.groupby('TEAM_ID')

# Shift IS_WIN et IS_HOME AVANT le calcul !
match_dataset['IS_WIN_SHIFTED'] = grouped['IS_WIN'].shift(1).fillna(0).astype(int)
match_dataset['IS_HOME_SHIFTED'] = grouped['IS_HOME'].shift(1).fillna(0).astype(int)

home_streaks = []
away_streaks = []
for _, df in grouped:
    home, away = calc_streak_home_away(df['IS_WIN_SHIFTED'].values, df['IS_HOME_SHIFTED'].values)
    home_streaks.extend(home)
    away_streaks.extend(away)

match_dataset['HOME_WIN_STREAK'] = home_streaks
match_dataset['AWAY_WIN_STREAK'] = away_streaks

# Clean up
match_dataset = match_dataset.drop(columns=['IS_WIN_SHIFTED', 'IS_HOME_SHIFTED'])



In [22]:
match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,ROLL_HOME_WINRATE_25,ROLL_AWAY_WINRATE_25,ROLL_HOME_WINRATE_50,ROLL_AWAY_WINRATE_50,ROLL_HOME_WINRATE_100,ROLL_AWAY_WINRATE_100,ROLL_HOME_WINRATE_200,ROLL_AWAY_WINRATE_200,HOME_WIN_STREAK,AWAY_WIN_STREAK
0,20000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,20000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,20000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,20000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,20000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64131,42400226,1610612760,2025-05-15,86.0,184.0,9.758,22.0,80.0,4.744,20.0,...,0.785714,0.818182,0.821429,0.772727,0.826923,0.770833,0.806122,0.656863,1,0
64132,42400216,1610612738,2025-05-16,62.0,172.0,9.404,24.0,80.0,6.732,14.0,...,0.636364,0.785714,0.666667,0.807692,0.705882,0.775510,0.764706,0.734694,1,0
64133,42400216,1610612752,2025-05-16,84.0,182.0,8.478,32.0,90.0,6.500,38.0,...,0.615385,0.583333,0.615385,0.625000,0.640000,0.600000,0.660000,0.560000,0,0
64134,42400227,1610612743,2025-05-18,66.0,168.0,8.318,20.0,88.0,5.016,34.0,...,0.538462,0.416667,0.653846,0.458333,0.607843,0.551020,0.712871,0.575758,0,0


##  Moyennes offensives/défensives home/away

In [23]:
for n in N_LIST:
    # Points marqués à domicile
    match_dataset[f'ROLL_HOME_PTS_FOR_{n}'] = (
        match_dataset
        .sort_values(['TEAM_ID', 'GAME_DATE'])
        .groupby('TEAM_ID')
        .apply(lambda df: df['PTS'].where(df['IS_HOME'] == 1).shift(1).rolling(n, min_periods=1).mean())
        .reset_index(level=0, drop=True)
    )
    # Points encaissés à domicile
    match_dataset[f'ROLL_HOME_PTS_AGAINST_{n}'] = (
        match_dataset
        .sort_values(['TEAM_ID', 'GAME_DATE'])
        .groupby('TEAM_ID')
        .apply(lambda df: df['OPP_PTS'].where(df['IS_HOME'] == 1).shift(1).rolling(n, min_periods=1).mean())
        .reset_index(level=0, drop=True)
    )
    # Idem pour l'extérieur
    match_dataset[f'ROLL_AWAY_PTS_FOR_{n}'] = (
        match_dataset
        .sort_values(['TEAM_ID', 'GAME_DATE'])
        .groupby('TEAM_ID')
        .apply(lambda df: df['PTS'].where(df['IS_HOME'] == 0).shift(1).rolling(n, min_periods=1).mean())
        .reset_index(level=0, drop=True)
    )
    match_dataset[f'ROLL_AWAY_PTS_AGAINST_{n}'] = (
        match_dataset
        .sort_values(['TEAM_ID', 'GAME_DATE'])
        .groupby('TEAM_ID')
        .apply(lambda df: df['OPP_PTS'].where(df['IS_HOME'] == 0).shift(1).rolling(n, min_periods=1).mean())
        .reset_index(level=0, drop=True)
    )


In [24]:
match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,ROLL_AWAY_PTS_FOR_50,ROLL_AWAY_PTS_AGAINST_50,ROLL_HOME_PTS_FOR_100,ROLL_HOME_PTS_AGAINST_100,ROLL_AWAY_PTS_FOR_100,ROLL_AWAY_PTS_AGAINST_100,ROLL_HOME_PTS_FOR_200,ROLL_HOME_PTS_AGAINST_200,ROLL_AWAY_PTS_FOR_200,ROLL_AWAY_PTS_AGAINST_200
0,20000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64131,42400226,1610612760,2025-05-15,86.0,184.0,9.758,22.0,80.0,4.744,20.0,...,234.545455,223.000000,243.730769,212.000000,230.500000,213.208333,264.367347,235.857143,244.470588,234.627451
64132,42400216,1610612738,2025-05-16,62.0,172.0,9.404,24.0,80.0,6.732,14.0,...,226.769231,208.769231,231.960784,213.333333,227.632653,212.040816,252.960784,227.607843,250.122449,234.632653
64133,42400216,1610612752,2025-05-16,84.0,182.0,8.478,32.0,90.0,6.500,38.0,...,219.333333,223.000000,233.520000,222.040000,223.920000,223.800000,243.480000,230.380000,239.820000,237.620000
64134,42400227,1610612743,2025-05-18,66.0,168.0,8.318,20.0,88.0,5.016,34.0,...,226.333333,237.666667,238.470588,228.666667,231.265306,232.326531,252.871287,237.366337,244.323232,244.282828


## Rolling Features on N matchs

In [25]:
for stat in ['PTS', 'REB', 'AST', 'FGM', 'FGA', 'FG_PCT', 'PLUS_MINUS']:
    for n in [3, 5, 10, 25, 50, 100,200]:
        match_dataset[f'ROLL_{stat}_{n}'] = (
            match_dataset
            .sort_values(['TEAM_ID', 'GAME_DATE'])
            .groupby('TEAM_ID')[stat]
            .transform(lambda x: x.shift(1).rolling(n, min_periods=1).mean())
        )


match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,ROLL_FG_PCT_50,ROLL_FG_PCT_100,ROLL_FG_PCT_200,ROLL_PLUS_MINUS_3,ROLL_PLUS_MINUS_5,ROLL_PLUS_MINUS_10,ROLL_PLUS_MINUS_25,ROLL_PLUS_MINUS_50,ROLL_PLUS_MINUS_100,ROLL_PLUS_MINUS_200
0,20000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64131,42400226,1610612760,2025-05-15,86.0,184.0,9.758,22.0,80.0,4.744,20.0,...,9.70400,9.61122,10.61244,10.0,88.0,135.0,118.8,125.6,124.0,94.77
64132,42400216,1610612738,2025-05-16,62.0,172.0,9.404,24.0,80.0,6.732,14.0,...,8.78824,8.62670,9.72085,130.0,70.0,99.0,94.8,90.6,85.7,102.60
64133,42400216,1610612752,2025-05-16,84.0,182.0,8.478,32.0,90.0,6.500,38.0,...,8.28408,8.29484,8.88282,-130.0,-70.0,-38.0,11.6,9.0,29.0,38.25
64134,42400227,1610612743,2025-05-18,66.0,168.0,8.318,20.0,88.0,5.016,34.0,...,8.65956,9.05864,10.02681,0.0,-68.0,-1.0,-14.4,6.4,22.4,39.25


## Streaks et Win Ratio

In [26]:
for n in [3, 5, 10, 25, 50, 100, 200]:
    match_dataset[f'ROLL_WIN_RATIO_{n}'] = (
        match_dataset
        .sort_values(['TEAM_ID', 'GAME_DATE'])
        .groupby('TEAM_ID')['IS_WIN']
        .transform(lambda x: x.shift(1).rolling(n, min_periods=1).mean())
    )


In [27]:
def calc_win_streak(results):
    streak = []
    cur = 0
    for r in results:
        if r == 1:
            cur += 1
        else:
            cur = 0
        streak.append(cur)
    return streak

# >>> Correction : shift IS_WIN avant de calculer le streak
match_dataset = match_dataset.sort_values(['TEAM_ID', 'GAME_DATE'])
match_dataset['IS_WIN_SHIFTED'] = (
    match_dataset
    .groupby('TEAM_ID')['IS_WIN']
    .shift(1)
    .fillna(0)
    .astype(int)
)

match_dataset['WIN_STREAK'] = (
    match_dataset
    .groupby('TEAM_ID')['IS_WIN_SHIFTED']
    .transform(calc_win_streak)
)



match_dataset = match_dataset.sort_values(['GAME_DATE','GAME_ID']).reset_index(drop=True)

In [28]:
match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,ROLL_PLUS_MINUS_200,ROLL_WIN_RATIO_3,ROLL_WIN_RATIO_5,ROLL_WIN_RATIO_10,ROLL_WIN_RATIO_25,ROLL_WIN_RATIO_50,ROLL_WIN_RATIO_100,ROLL_WIN_RATIO_200,IS_WIN_SHIFTED,WIN_STREAK
0,20000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,20000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
2,20000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
3,20000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
4,20000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64131,42400226,1610612760,2025-05-15,86.0,184.0,9.758,22.0,80.0,4.744,20.0,...,94.77,0.666667,0.6,0.8,0.80,0.80,0.80,0.730,1,2
64132,42400216,1610612738,2025-05-16,62.0,172.0,9.404,24.0,80.0,6.732,14.0,...,102.60,0.666667,0.4,0.6,0.72,0.74,0.74,0.750,1,1
64133,42400216,1610612752,2025-05-16,84.0,182.0,8.478,32.0,90.0,6.500,38.0,...,38.25,0.333333,0.6,0.6,0.60,0.62,0.62,0.610,0,0
64134,42400227,1610612743,2025-05-18,66.0,168.0,8.318,20.0,88.0,5.016,34.0,...,39.25,0.333333,0.4,0.6,0.48,0.56,0.58,0.645,1,1


## Rest Days & Advantage

In [29]:
match_dataset = match_dataset.sort_values(['TEAM_ID', 'GAME_DATE'])
match_dataset['DAYS_SINCE_LAST_GAME'] = (
    match_dataset.groupby('TEAM_ID')['GAME_DATE'].diff().dt.days.fillna(7) # Si NaN = 7 jours par défaut
)

match_dataset['OPP_DAYS_SINCE_LAST_GAME'] = (
    match_dataset.groupby('OPP_TEAM_ID')['GAME_DATE'].diff().dt.days.fillna(7)
)
match_dataset['REST_ADVANTAGE'] = match_dataset['DAYS_SINCE_LAST_GAME'] - match_dataset['OPP_DAYS_SINCE_LAST_GAME']
ch_dataset = match_dataset.sort_values(['TEAM_ID', 'GAME_DATE'])
match_dataset['DAYS_SINCE_LAST_GAME'] = (
    match_dataset.groupby('TEAM_ID')['GAME_DATE'].diff().dt.days.fillna(7) # Si NaN = 7 jours par défaut
)

match_dataset['OPP_DAYS_SINCE_LAST_GAME'] = (
    match_dataset.groupby('OPP_TEAM_ID')['GAME_DATE'].diff().dt.days.fillna(7)
)
match_dataset['REST_ADVANTAGE'] = match_dataset['DAYS_SINCE_LAST_GAME'] - match_dataset['OPP_DAYS_SINCE_LAST_GAME']


In [30]:
#group again match_dataset by game_id and game_date

match_dataset = match_dataset.sort_values(['GAME_DATE', 'GAME_ID']).reset_index(drop=True)
match_dataset


,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,ROLL_WIN_RATIO_10,ROLL_WIN_RATIO_25,ROLL_WIN_RATIO_50,ROLL_WIN_RATIO_100,ROLL_WIN_RATIO_200,IS_WIN_SHIFTED,WIN_STREAK,DAYS_SINCE_LAST_GAME,OPP_DAYS_SINCE_LAST_GAME,REST_ADVANTAGE
0,20000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,NaN,NaN,NaN,NaN,NaN,0,0,7.0,-8880.0,8887.0
1,20000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,NaN,NaN,NaN,NaN,NaN,0,0,7.0,-8869.0,8876.0
2,20000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,NaN,NaN,NaN,NaN,NaN,0,0,7.0,-8904.0,8911.0
3,20000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,NaN,NaN,NaN,NaN,NaN,0,0,7.0,-8868.0,8875.0
4,20000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,NaN,NaN,NaN,NaN,NaN,0,0,7.0,-8908.0,8915.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64131,42400226,1610612760,2025-05-15,86.0,184.0,9.758,22.0,80.0,4.744,20.0,...,0.8,0.80,0.80,0.80,0.730,1,2,2.0,2.0,0.0
64132,42400216,1610612738,2025-05-16,62.0,172.0,9.404,24.0,80.0,6.732,14.0,...,0.6,0.72,0.74,0.74,0.750,1,1,2.0,2.0,0.0
64133,42400216,1610612752,2025-05-16,84.0,182.0,8.478,32.0,90.0,6.500,38.0,...,0.6,0.60,0.62,0.62,0.610,0,0,2.0,2.0,0.0
64134,42400227,1610612743,2025-05-18,66.0,168.0,8.318,20.0,88.0,5.016,34.0,...,0.6,0.48,0.56,0.58,0.645,1,1,3.0,3.0,0.0


## Rest advantage home/away

In [31]:
for n in N_LIST:
    match_dataset[f'ROLL_HOME_REST_ADV_{n}'] = (
        match_dataset
        .sort_values(['TEAM_ID', 'GAME_DATE'])
        .groupby('TEAM_ID')
        .apply(lambda df: df['REST_ADVANTAGE'].where(df['IS_HOME'] == 1).shift(1).rolling(n, min_periods=1).mean())
        .reset_index(level=0, drop=True)
    )
    match_dataset[f'ROLL_AWAY_REST_ADV_{n}'] = (
        match_dataset
        .sort_values(['TEAM_ID', 'GAME_DATE'])
        .groupby('TEAM_ID')
        .apply(lambda df: df['REST_ADVANTAGE'].where(df['IS_HOME'] == 0).shift(1).rolling(n, min_periods=1).mean())
        .reset_index(level=0, drop=True)
    )


In [32]:
match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,ROLL_HOME_REST_ADV_10,ROLL_AWAY_REST_ADV_10,ROLL_HOME_REST_ADV_25,ROLL_AWAY_REST_ADV_25,ROLL_HOME_REST_ADV_50,ROLL_AWAY_REST_ADV_50,ROLL_HOME_REST_ADV_100,ROLL_AWAY_REST_ADV_100,ROLL_HOME_REST_ADV_200,ROLL_AWAY_REST_ADV_200
0,20000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64131,42400226,1610612760,2025-05-15,86.0,184.0,9.758,22.0,80.0,4.744,20.0,...,-12.0,-12.400000,-43.500000,-66.636364,-67.821429,-58.590909,-109.903846,-96.937500,-122.377551,-97.411765
64132,42400216,1610612738,2025-05-16,62.0,172.0,9.404,24.0,80.0,6.732,14.0,...,-3.5,0.000000,-19.909091,-64.214286,-66.583333,-77.692308,-102.843137,-95.183673,-97.392157,-107.602041
64133,42400216,1610612752,2025-05-16,84.0,182.0,8.478,32.0,90.0,6.500,38.0,...,0.0,-3.833333,-41.307692,-31.500000,-90.384615,-74.083333,-95.820000,-108.760000,-93.080000,-117.270000
64134,42400227,1610612743,2025-05-18,66.0,168.0,8.318,20.0,88.0,5.016,34.0,...,0.0,-10.800000,-30.692308,-25.583333,-33.307692,-96.583333,-88.764706,-116.265306,-90.782178,-111.575758


## H2H - Head To Head WinRate (Last N games)

In [33]:
import pandas as pd
from collections import defaultdict, deque

# On suppose que match_dataset contient déjà une ligne par équipe/match (après le merge et le tri)
# Et que tu as bien trié par date/match
match_dataset = match_dataset.sort_values(['GAME_DATE', 'GAME_ID']).reset_index(drop=True)

# On prépare les features H2H rolling pour chaque équipe face à chaque adversaire
N_LIST = [5, 10, 25, 50, 100, 200]  # tu peux réduire à ce que tu veux
for N in N_LIST:
    # Dictionnaire: (team, opponent) -> deque des derniers résultats (1 = win, 0 = lose)
    last_n_results = defaultdict(lambda: deque(maxlen=N))
    h2h_diff_list = []
    h2h_winrate_list = []
    h2h_count_list = []

    # Pour chaque ligne (match), on met à jour le rolling H2H
    for idx, row in match_dataset.iterrows():
        team = row['TEAM_ID']
        opp = row['OPP_TEAM_ID']
        team_pts = row['PTS']
        opp_pts = row['OPP_PTS']

        # Calcul du résultat précédent
        history = last_n_results[(team, opp)]
        n_prev = len(history)
        winrate = sum(history) / n_prev if n_prev > 0 else 0.5
        diff = sum(history) - (n_prev - sum(history)) if n_prev > 0 else 0  # nb_victoires - nb_défaites

        h2h_diff_list.append(diff)
        h2h_winrate_list.append(winrate)
        h2h_count_list.append(n_prev)

        # Maj du résultat courant (après l'utilisation pour ne pas polluer la ligne actuelle)
        last_n_results[(team, opp)].append(1 if team_pts > opp_pts else 0)

    match_dataset[f'H2H_LAST_{N}_DIFF'] = h2h_diff_list
    match_dataset[f'H2H_LAST_{N}_WINRATE'] = h2h_winrate_list
    match_dataset[f'H2H_LAST_{N}_COUNT'] = h2h_count_list



In [34]:
match_dataset.tail(10)

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,H2H_LAST_25_COUNT,H2H_LAST_50_DIFF,H2H_LAST_50_WINRATE,H2H_LAST_50_COUNT,H2H_LAST_100_DIFF,H2H_LAST_100_WINRATE,H2H_LAST_100_COUNT,H2H_LAST_200_DIFF,H2H_LAST_200_WINRATE,H2H_LAST_200_COUNT
64126,42400215,1610612738,2025-05-14,88.0,168.0,12.368,44.0,98.0,8.458,34.0,...,25,14,0.64,50,32,0.660000,100,37,0.663717,113
64127,42400215,1610612752,2025-05-14,58.0,162.0,6.822,24.0,60.0,4.344,64.0,...,25,-14,0.36,50,-32,0.340000,100,-37,0.336283,113
64128,42400235,1610612744,2025-05-14,78.0,180.0,9.014,22.0,78.0,4.168,42.0,...,25,12,0.62,50,16,0.585106,94,16,0.585106,94
64129,42400235,1610612750,2025-05-14,98.0,156.0,9.868,26.0,62.0,4.112,20.0,...,25,-12,0.38,50,-16,0.414894,94,-16,0.414894,94
64130,42400226,1610612743,2025-05-15,80.0,172.0,6.872,24.0,64.0,5.132,54.0,...,25,0,0.50,50,4,0.520000,100,5,0.523364,107
64131,42400226,1610612760,2025-05-15,86.0,184.0,9.758,22.0,80.0,4.744,20.0,...,25,0,0.50,50,-4,0.480000,100,-5,0.476636,107
64132,42400216,1610612738,2025-05-16,62.0,172.0,9.404,24.0,80.0,6.732,14.0,...,25,14,0.64,50,34,0.670000,100,38,0.666667,114
64133,42400216,1610612752,2025-05-16,84.0,182.0,8.478,32.0,90.0,6.500,38.0,...,25,-14,0.36,50,-34,0.330000,100,-38,0.333333,114
64134,42400227,1610612743,2025-05-18,66.0,168.0,8.318,20.0,88.0,5.016,34.0,...,25,2,0.52,50,4,0.520000,100,6,0.527778,108
64135,42400227,1610612760,2025-05-18,94.0,192.0,11.902,24.0,78.0,8.200,38.0,...,25,-2,0.48,50,-4,0.480000,100,-6,0.472222,108


## H2H points diff (N last games)

In [35]:
for N in N_LIST:
    last_n_pts_for = defaultdict(lambda: deque(maxlen=N))
    last_n_pts_against = defaultdict(lambda: deque(maxlen=N))
    pts_for_list = []
    pts_against_list = []
    margin_list = []

    for idx, row in match_dataset.iterrows():
        team = row['TEAM_ID']
        opp = row['OPP_TEAM_ID']
        team_pts = row['PTS']
        opp_pts = row['OPP_PTS']

        history_for = last_n_pts_for[(team, opp)]
        history_against = last_n_pts_against[(team, opp)]
        n_prev = len(history_for)

        avg_for = sum(history_for)/n_prev if n_prev > 0 else 0
        avg_against = sum(history_against)/n_prev if n_prev > 0 else 0
        avg_margin = (sum(history_for) - sum(history_against))/n_prev if n_prev > 0 else 0

        pts_for_list.append(avg_for)
        pts_against_list.append(avg_against)
        margin_list.append(avg_margin)

        # Update with the current match AFTER calculation
        last_n_pts_for[(team, opp)].append(team_pts)
        last_n_pts_against[(team, opp)].append(opp_pts)

    match_dataset[f'H2H_LAST_{N}_PTS_FOR'] = pts_for_list
    match_dataset[f'H2H_LAST_{N}_PTS_AGAINST'] = pts_against_list
    match_dataset[f'H2H_LAST_{N}_MARGIN'] = margin_list


In [36]:
match_dataset



#print all columns
 #print(match_dataset.columns)

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,H2H_LAST_25_MARGIN,H2H_LAST_50_PTS_FOR,H2H_LAST_50_PTS_AGAINST,H2H_LAST_50_MARGIN,H2H_LAST_100_PTS_FOR,H2H_LAST_100_PTS_AGAINST,H2H_LAST_100_MARGIN,H2H_LAST_200_PTS_FOR,H2H_LAST_200_PTS_AGAINST,H2H_LAST_200_MARGIN
0,20000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.000000,0.000000,0.000000
1,20000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.000000,0.000000,0.000000
2,20000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.000000,0.000000,0.000000
3,20000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.000000,0.000000,0.000000
4,20000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64131,42400226,1610612760,2025-05-15,86.0,184.0,9.758,22.0,80.0,4.744,20.0,...,0.40,221.48,217.68,3.80,215.82,215.70,0.12,215.327103,214.635514,0.691589
64132,42400216,1610612738,2025-05-16,62.0,172.0,9.404,24.0,80.0,6.732,14.0,...,11.04,224.48,212.92,11.56,212.00,202.38,9.62,209.894737,200.421053,9.473684
64133,42400216,1610612752,2025-05-16,84.0,182.0,8.478,32.0,90.0,6.500,38.0,...,-11.04,212.92,224.48,-11.56,202.38,212.00,-9.62,200.421053,209.894737,-9.473684
64134,42400227,1610612743,2025-05-18,66.0,168.0,8.318,20.0,88.0,5.016,34.0,...,-0.08,218.72,221.56,-2.84,216.32,216.26,0.06,214.851852,215.314815,-0.462963


In [37]:
print(df_games[['GAME_ID', 'TEAM_ID']].duplicated().sum())  # doit donner 0

0


## H2H Rolling par Saison

In [38]:
# Assure-toi d’avoir une colonne SEASON dans match_dataset, sinon merge avec df_games
if 'SEASON' not in match_dataset.columns:
    # Tu peux ajouter la colonne depuis df_games
    match_dataset = match_dataset.merge(
        df_games[['GAME_ID', 'TEAM_ID', 'SEASON']].drop_duplicates(),
        on=['GAME_ID', 'TEAM_ID'],
        how='left'
    )



# Création d’un compteur par saison
h2h_season_wins = defaultdict(int)
h2h_season_matches = defaultdict(int)
season_win_counts = []
season_match_counts = []

for idx, row in match_dataset.iterrows():
    season = row['SEASON']
    team = row['TEAM_ID']
    opp = row['OPP_TEAM_ID']
    team_pts = row['PTS']
    opp_pts = row['OPP_PTS']

    win_count = h2h_season_wins[(team, opp, season)]
    match_count = h2h_season_matches[(team, opp, season)]

    season_win_counts.append(win_count)
    season_match_counts.append(match_count)

    # Update counters
    h2h_season_matches[(team, opp, season)] += 1
    if team_pts > opp_pts:
        h2h_season_wins[(team, opp, season)] += 1

match_dataset["H2H_SEASON_WINS"] = season_win_counts
match_dataset["H2H_SEASON_MATCHES"] = season_match_counts
match_dataset["H2H_SEASON_WINRATE"] = [
    w / m if m > 0 else 0 for w, m in zip(season_win_counts, season_match_counts)
]


In [39]:
match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,H2H_LAST_100_PTS_FOR,H2H_LAST_100_PTS_AGAINST,H2H_LAST_100_MARGIN,H2H_LAST_200_PTS_FOR,H2H_LAST_200_PTS_AGAINST,H2H_LAST_200_MARGIN,SEASON,H2H_SEASON_WINS,H2H_SEASON_MATCHES,H2H_SEASON_WINRATE
0,20000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,0.00,0.00,0.00,0.000000,0.000000,0.000000,2000-01,0,0,0.000000
1,20000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,0.00,0.00,0.00,0.000000,0.000000,0.000000,2000-01,0,0,0.000000
2,20000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,0.00,0.00,0.00,0.000000,0.000000,0.000000,2000-01,0,0,0.000000
3,20000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,0.00,0.00,0.00,0.000000,0.000000,0.000000,2000-01,0,0,0.000000
4,20000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,0.00,0.00,0.00,0.000000,0.000000,0.000000,2000-01,0,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64131,42400226,1610612760,2025-05-15,86.0,184.0,9.758,22.0,80.0,4.744,20.0,...,215.82,215.70,0.12,215.327103,214.635514,0.691589,2024-25,5,9,0.555556
64132,42400216,1610612738,2025-05-16,62.0,172.0,9.404,24.0,80.0,6.732,14.0,...,212.00,202.38,9.62,209.894737,200.421053,9.473684,2024-25,6,9,0.666667
64133,42400216,1610612752,2025-05-16,84.0,182.0,8.478,32.0,90.0,6.500,38.0,...,202.38,212.00,-9.62,200.421053,209.894737,-9.473684,2024-25,3,9,0.333333
64134,42400227,1610612743,2025-05-18,66.0,168.0,8.318,20.0,88.0,5.016,34.0,...,216.32,216.26,0.06,214.851852,215.314815,-0.462963,2024-25,5,10,0.500000


## H2H Streaks

In [40]:
h2h_streaks = defaultdict(int)
current_streak = defaultdict(int)
last_result = defaultdict(lambda: None)
streak_list = []

for idx, row in match_dataset.iterrows():
    team = row['TEAM_ID']
    opp = row['OPP_TEAM_ID']
    team_pts = row['PTS']
    opp_pts = row['OPP_PTS']

    key = (team, opp)

    # Récupère le streak précédent
    streak = current_streak[key]
    last = last_result[key]

    streak_list.append(streak)

    # Update: Si victoire, on incrémente, sinon on reset à 0
    if team_pts > opp_pts:
        if last == "W":
            current_streak[key] += 1
        else:
            current_streak[key] = 1
        last_result[key] = "W"
    else:
        current_streak[key] = 0
        last_result[key] = "L"

match_dataset['H2H_WIN_STREAK'] = streak_list
match_dataset


,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,H2H_LAST_100_PTS_AGAINST,H2H_LAST_100_MARGIN,H2H_LAST_200_PTS_FOR,H2H_LAST_200_PTS_AGAINST,H2H_LAST_200_MARGIN,SEASON,H2H_SEASON_WINS,H2H_SEASON_MATCHES,H2H_SEASON_WINRATE,H2H_WIN_STREAK
0,20000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,0.00,0.00,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0
1,20000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,0.00,0.00,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0
2,20000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,0.00,0.00,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0
3,20000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,0.00,0.00,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0
4,20000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,0.00,0.00,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64131,42400226,1610612760,2025-05-15,86.0,184.0,9.758,22.0,80.0,4.744,20.0,...,215.70,0.12,215.327103,214.635514,0.691589,2024-25,5,9,0.555556,2
64132,42400216,1610612738,2025-05-16,62.0,172.0,9.404,24.0,80.0,6.732,14.0,...,202.38,9.62,209.894737,200.421053,9.473684,2024-25,6,9,0.666667,1
64133,42400216,1610612752,2025-05-16,84.0,182.0,8.478,32.0,90.0,6.500,38.0,...,212.00,-9.62,200.421053,209.894737,-9.473684,2024-25,3,9,0.333333,0
64134,42400227,1610612743,2025-05-18,66.0,168.0,8.318,20.0,88.0,5.016,34.0,...,216.26,0.06,214.851852,215.314815,-0.462963,2024-25,5,10,0.500000,1


## Calcul du ELO global

In [41]:
# --- ELO GLOBAL CALCULATION ---

from collections import defaultdict

# 1. Trie les matchs par date pour garder la cohérence chronologique
match_dataset = match_dataset.sort_values(by=['GAME_DATE', 'GAME_ID', 'TEAM_ID']).reset_index(drop=True)

# 2. Paramètres du ELO
elo_start = 1500
k_factor = 24

# 3. Dictionnaire pour stocker l'ELO de chaque équipe
elo_history = defaultdict(lambda: elo_start)
elo_hist_list = []

# 4. Calcul de l'ELO avant chaque match (pour chaque équipe)
for idx, row in match_dataset.iterrows():
    team = row['TEAM_ID']
    opp = row['OPP_TEAM_ID']
    game_id = row['GAME_ID']
    
    team_elo_pre = elo_history[team]
    opp_elo_pre = elo_history[opp]
    
    # Stocke les valeurs PRE-match (avant update)
    elo_hist_list.append({
        'GAME_ID': game_id,
        'TEAM_ID': team,
        'ELO_PRE': team_elo_pre
    })
    
    # Calcul du résultat du match
    # On considère 1 pour victoire, 0 pour défaite (pas de draw NBA)
    team_pts = row['PTS']
    opp_pts = row['OPP_PTS']
    if team_pts > opp_pts:
        outcome = 1
    else:
        outcome = 0
    
    # Probabilité attendue de victoire
    expected = 1 / (1 + 10 ** ((opp_elo_pre - team_elo_pre) / 400))
    
    # MAJ de l'ELO de l'équipe après le match
    new_elo = team_elo_pre + k_factor * (outcome - expected)
    elo_history[team] = new_elo

# 5. Crée un DataFrame des ELO_PRE par équipe et par match
elo_df = pd.DataFrame(elo_hist_list)

# 6. Ajoute ELO_PRE pour chaque équipe
match_dataset = match_dataset.merge(
    elo_df,
    on=['GAME_ID', 'TEAM_ID'],
    how='left'
)

# 7. Ajoute ELO_PRE de l'adversaire (OPP_ELO_PRE) en croisant sur OPP_TEAM_ID
match_dataset = match_dataset.merge(
    elo_df.rename(columns={'TEAM_ID': 'OPP_TEAM_ID', 'ELO_PRE': 'OPP_ELO_PRE'}),
    on=['GAME_ID', 'OPP_TEAM_ID'],
    how='left'
)

# 8. (Optionnel) Vérification rapide
print(match_dataset[['GAME_ID', 'TEAM_ID', 'ELO_PRE', 'OPP_TEAM_ID', 'OPP_ELO_PRE']].tail(10))


        GAME_ID     TEAM_ID      ELO_PRE  OPP_TEAM_ID  OPP_ELO_PRE
64126  42400215  1610612738  1678.105711   1610612752  1633.955322
64127  42400215  1610612752  1633.955322   1610612738  1678.105711
64128  42400235  1610612744  1590.203635   1610612750  1654.079481
64129  42400235  1610612750  1654.079481   1610612744  1590.203635
64130  42400226  1610612743  1610.859999   1610612760  1777.217635
64131  42400226  1610612760  1777.217635   1610612743  1610.859999
64132  42400216  1610612738  1688.588966   1610612752  1623.826901
64133  42400216  1610612752  1623.826901   1610612738  1688.588966
64134  42400227  1610612743  1628.203536   1610612760  1760.364845
64135  42400227  1610612760  1760.364845   1610612743  1628.203536


In [42]:
match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,H2H_LAST_200_PTS_FOR,H2H_LAST_200_PTS_AGAINST,H2H_LAST_200_MARGIN,SEASON,H2H_SEASON_WINS,H2H_SEASON_MATCHES,H2H_SEASON_WINRATE,H2H_WIN_STREAK,ELO_PRE,OPP_ELO_PRE
0,20000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0,1500.000000,1500.000000
1,20000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0,1500.000000,1500.000000
2,20000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0,1500.000000,1500.000000
3,20000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0,1500.000000,1500.000000
4,20000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,0.000000,0.000000,0.000000,2000-01,0,0,0.000000,0,1500.000000,1500.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64131,42400226,1610612760,2025-05-15,86.0,184.0,9.758,22.0,80.0,4.744,20.0,...,215.327103,214.635514,0.691589,2024-25,5,9,0.555556,2,1777.217635,1610.859999
64132,42400216,1610612738,2025-05-16,62.0,172.0,9.404,24.0,80.0,6.732,14.0,...,209.894737,200.421053,9.473684,2024-25,6,9,0.666667,1,1688.588966,1623.826901
64133,42400216,1610612752,2025-05-16,84.0,182.0,8.478,32.0,90.0,6.500,38.0,...,200.421053,209.894737,-9.473684,2024-25,3,9,0.333333,0,1623.826901,1688.588966
64134,42400227,1610612743,2025-05-18,66.0,168.0,8.318,20.0,88.0,5.016,34.0,...,214.851852,215.314815,-0.462963,2024-25,5,10,0.500000,1,1628.203536,1760.364845


## Calcul du ELO par saison

In [43]:
from collections import defaultdict

# On part d'un dataset déjà trié par GAME_DATE, SEASON_ID, GAME_ID, TEAM_ID
match_dataset = match_dataset.sort_values(by=['SEASON', 'GAME_DATE', 'GAME_ID', 'TEAM_ID']).reset_index(drop=True)

elo_start = 1500
k_factor = 24

elo_history_season = defaultdict(lambda: elo_start)
elo_hist_list_season = []

for idx, row in match_dataset.iterrows():
    season = row['SEASON']
    team = row['TEAM_ID']
    opp = row['OPP_TEAM_ID']
    game_id = row['GAME_ID']
    
    # Clés différentes pour chaque saison
    key_team = (season, team)
    key_opp = (season, opp)
    
    team_elo_pre = elo_history_season[key_team]
    opp_elo_pre = elo_history_season[key_opp]
    
    # Stocke avant update
    elo_hist_list_season.append({
        'GAME_ID': game_id,
        'TEAM_ID': team,
        'SEASON': season,
        'ELO_PRE_SEASON': team_elo_pre
    })
    
    team_pts = row['PTS']
    opp_pts = row['OPP_PTS']
    outcome = 1 if team_pts > opp_pts else 0
    expected = 1 / (1 + 10 ** ((opp_elo_pre - team_elo_pre) / 400))
    new_elo = team_elo_pre + k_factor * (outcome - expected)
    elo_history_season[key_team] = new_elo

elo_season_df = pd.DataFrame(elo_hist_list_season)

# Ajoute ELO_PRE_SEASON pour chaque équipe
match_dataset = match_dataset.merge(
    elo_season_df,
    on=['GAME_ID', 'TEAM_ID', 'SEASON'],
    how='left'
)

# Ajoute ELO_PRE_SEASON de l'adversaire (OPP_ELO_PRE_SEASON)
match_dataset = match_dataset.merge(
    elo_season_df.rename(columns={'TEAM_ID': 'OPP_TEAM_ID', 'ELO_PRE_SEASON': 'OPP_ELO_PRE_SEASON'}),
    on=['GAME_ID', 'OPP_TEAM_ID', 'SEASON'],
    how='left'
)

# Vérification rapide
print(match_dataset[['GAME_ID', 'SEASON', 'TEAM_ID', 'ELO_PRE_SEASON', 'OPP_TEAM_ID', 'OPP_ELO_PRE_SEASON']].tail(10))


        GAME_ID   SEASON     TEAM_ID  ELO_PRE_SEASON  OPP_TEAM_ID  \
64126  42400215  2024-25  1610612738     1652.618344   1610612752   
64127  42400215  2024-25  1610612752     1627.587981   1610612738   
64128  42400235  2024-25  1610612744     1580.648482   1610612750   
64129  42400235  2024-25  1610612750     1639.353297   1610612744   
64130  42400226  2024-25  1610612743     1594.103530   1610612760   
64131  42400226  2024-25  1610612760     1756.109250   1610612743   
64132  42400216  2024-25  1610612738     1663.755318   1610612752   
64133  42400216  2024-25  1610612752     1616.832664   1610612738   
64134  42400227  2024-25  1610612743     1611.325892   1610612760   
64135  42400227  2024-25  1610612760     1739.379287   1610612743   

       OPP_ELO_PRE_SEASON  
64126         1627.587981  
64127         1652.618344  
64128         1639.353297  
64129         1580.648482  
64130         1756.109250  
64131         1594.103530  
64132         1616.832664  
64133         16

In [44]:
match_dataset

,GAME_ID,TEAM_ID,GAME_DATE,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,...,H2H_LAST_200_MARGIN,SEASON,H2H_SEASON_WINS,H2H_SEASON_MATCHES,H2H_SEASON_WINRATE,H2H_WIN_STREAK,ELO_PRE,OPP_ELO_PRE,ELO_PRE_SEASON,OPP_ELO_PRE_SEASON
0,20000001,1610612752,2000-10-31,50.0,140.0,8.906,6.0,22.0,2.666,38.0,...,0.000000,2000-01,0,0,0.000000,0,1500.000000,1500.000000,1500.000000,1500.000000
1,20000001,1610612755,2000-10-31,76.0,132.0,10.382,6.0,16.0,3.000,44.0,...,0.000000,2000-01,0,0,0.000000,0,1500.000000,1500.000000,1500.000000,1500.000000
2,20000002,1610612739,2000-10-31,64.0,156.0,6.796,4.0,14.0,3.000,40.0,...,0.000000,2000-01,0,0,0.000000,0,1500.000000,1500.000000,1500.000000,1500.000000
3,20000002,1610612751,2000-10-31,62.0,170.0,6.524,6.0,20.0,2.334,34.0,...,0.000000,2000-01,0,0,0.000000,0,1500.000000,1500.000000,1500.000000,1500.000000
4,20000003,1610612753,2000-10-31,68.0,158.0,7.900,12.0,32.0,2.750,46.0,...,0.000000,2000-01,0,0,0.000000,0,1500.000000,1500.000000,1500.000000,1500.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
64131,42400226,1610612760,2025-05-15,86.0,184.0,9.758,22.0,80.0,4.744,20.0,...,0.691589,2024-25,5,9,0.555556,2,1777.217635,1610.859999,1756.109250,1594.103530
64132,42400216,1610612738,2025-05-16,62.0,172.0,9.404,24.0,80.0,6.732,14.0,...,9.473684,2024-25,6,9,0.666667,1,1688.588966,1623.826901,1663.755318,1616.832664
64133,42400216,1610612752,2025-05-16,84.0,182.0,8.478,32.0,90.0,6.500,38.0,...,-9.473684,2024-25,3,9,0.333333,0,1623.826901,1688.588966,1616.832664,1663.755318
64134,42400227,1610612743,2025-05-18,66.0,168.0,8.318,20.0,88.0,5.016,34.0,...,-0.462963,2024-25,5,10,0.500000,1,1628.203536,1760.364845,1611.325892,1739.379287


## Save final dataset in CSV 

In [45]:

# Chemin de sauvegarde (modifie selon ta structure de dossiers)
final_date = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
final_path = os.path.join(DATA_FINAL_DATASET_DIR, f'nba_features_final_{final_date}.csv')

match_dataset.to_csv(final_path, index=False)
print(f'✅ Dataset sauvegardé à {final_path}')


✅ Dataset sauvegardé à data/final_dataset/nba_features_final_2025-05-23_11-59-28.csv


## Clean final dataset for training and predictions

In [46]:
# Liste des colonnes à dropper (toutes les stats brutes et colonnes de match, identifiants inutiles, etc.)
drop_cols = [
    # Identifiants et logs
    "GAME_ID", "GAME_DATE", "OPP_TEAM_ID", "OPP_GAME_DATE",

    # Stats brutes de match (pour les deux équipes)
    "FGM", "FGA", "FG_PCT", "FG3M", "FG3A", "FG3_PCT", "FTM", "FTA", "FT_PCT",
    "OREB", "DREB", "REB", "AST", "STL", "BLK", "TO", "PF", "PTS", "PLUS_MINUS", "MINUTES_PLAYED",

    "OPP_FGM", "OPP_FGA", "OPP_FG_PCT", "OPP_FG3M", "OPP_FG3A", "OPP_FG3_PCT", "OPP_FTM", "OPP_FTA", "OPP_FT_PCT",
    "OPP_OREB", "OPP_DREB", "OPP_REB", "OPP_AST", "OPP_STL", "OPP_BLK", "OPP_TO", "OPP_PF", "OPP_PTS",
    "OPP_PLUS_MINUS", "OPP_MINUTES_PLAYED",
    "POINT_DIFF"
]

# Droppage effectif
final_dataset = match_dataset.drop(columns=[col for col in drop_cols if col in match_dataset.columns])

# Sauvegarde finale
final_cleaned_path = os.path.join(DATA_FINAL_CLEANED_DATASET_DIR, f'nba_features_cleaned_final_{final_date}.csv')

final_dataset.to_csv(final_cleaned_path, index=False)
print(f'✅ Dataset sauvegardé à {final_cleaned_path}')


✅ Dataset sauvegardé à data/final_cleaned_dataset/nba_features_cleaned_final_2025-05-23_11-59-28.csv


In [47]:
for col in final_dataset.columns:
    print(f"{col}: {final_dataset[col].isnull().sum()}")  # Affiche le nombre de NaN par colonne

TEAM_ID: 0
IS_WIN: 0
IS_HOME: 0
ROLL_HOME_WINRATE_5: 1701
ROLL_AWAY_WINRATE_5: 1393
ROLL_HOME_WINRATE_10: 54
ROLL_AWAY_WINRATE_10: 53
ROLL_HOME_WINRATE_25: 52
ROLL_AWAY_WINRATE_25: 53
ROLL_HOME_WINRATE_50: 52
ROLL_AWAY_WINRATE_50: 53
ROLL_HOME_WINRATE_100: 52
ROLL_AWAY_WINRATE_100: 53
ROLL_HOME_WINRATE_200: 52
ROLL_AWAY_WINRATE_200: 53
HOME_WIN_STREAK: 0
AWAY_WIN_STREAK: 0
ROLL_HOME_PTS_FOR_5: 1701
ROLL_HOME_PTS_AGAINST_5: 1701
ROLL_AWAY_PTS_FOR_5: 1393
ROLL_AWAY_PTS_AGAINST_5: 1393
ROLL_HOME_PTS_FOR_10: 54
ROLL_HOME_PTS_AGAINST_10: 54
ROLL_AWAY_PTS_FOR_10: 53
ROLL_AWAY_PTS_AGAINST_10: 53
ROLL_HOME_PTS_FOR_25: 52
ROLL_HOME_PTS_AGAINST_25: 52
ROLL_AWAY_PTS_FOR_25: 53
ROLL_AWAY_PTS_AGAINST_25: 53
ROLL_HOME_PTS_FOR_50: 52
ROLL_HOME_PTS_AGAINST_50: 52
ROLL_AWAY_PTS_FOR_50: 53
ROLL_AWAY_PTS_AGAINST_50: 53
ROLL_HOME_PTS_FOR_100: 52
ROLL_HOME_PTS_AGAINST_100: 52
ROLL_AWAY_PTS_FOR_100: 53
ROLL_AWAY_PTS_AGAINST_100: 53
ROLL_HOME_PTS_FOR_200: 52
ROLL_HOME_PTS_AGAINST_200: 52
ROLL_AWAY_PTS_FOR_20

In [ ]:
#store end time of notebook
end_time = datetime.datetime.now()
print("End time: ", end_time)
print("Total time: ", end_time - start_time)

End time:  2025-05-23 11:59:48.676326
Total time:  0:04:39.265134


: 